# Granular Category EDA

Exploratory analysis of `granular_category` (~101 values). Examines category distributions,
relationships with redaction metrics, and document length patterns.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
PALETTE = "Set2"
DEFAULT_FIGSIZE = (10, 6)
WIDE_FIGSIZE = (14, 5)
HIST_KWARGS = dict(bins=30, edgecolor="black", alpha=0.7)

In [ ]:
DOC_PATH = "../datasets/document_level.csv"
LOOKUP_PATH = "../data/lookup_table.csv"

doc_df = pd.read_csv(DOC_PATH)
lookup_df = pd.read_csv(LOOKUP_PATH)

df = doc_df.merge(
    lookup_df[["folder_name", "granular_category"]],
    on="folder_name",
    how="left",
)
print(f"document_level shape: {doc_df.shape}")
print(f"merged shape:         {df.shape}")
print(f"unique categories:    {df['granular_category'].nunique()}")
print()
df.head()

## Distribution

In [ ]:
# Horizontal bar chart of ALL granular categories (sorted by count)
cat_counts = df["granular_category"].value_counts()

fig, ax = plt.subplots(figsize=(12, 20))
cat_counts.iloc[::-1].plot.barh(ax=ax, color=sns.color_palette(PALETTE)[0], edgecolor="black")
ax.set_title("Document Counts by Granular Category (All)")
ax.set_xlabel("Number of Documents")
ax.set_ylabel("Granular Category")
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 categories bar chart
top20 = cat_counts.head(20)

fig, ax = plt.subplots(figsize=DEFAULT_FIGSIZE)
top20.iloc[::-1].plot.barh(ax=ax, color=sns.color_palette(PALETTE)[1], edgecolor="black")
ax.set_title("Top 20 Granular Categories by Document Count")
ax.set_xlabel("Number of Documents")
ax.set_ylabel("Granular Category")
plt.tight_layout()
plt.show()

In [ ]:
# Summary stats table: counts and percentages
summary = pd.DataFrame({
    "count": cat_counts,
    "pct": (cat_counts / cat_counts.sum() * 100).round(2),
})
summary.index.name = "granular_category"
print(f"Total documents: {cat_counts.sum()}")
print(f"Total categories: {len(cat_counts)}")
print()
summary

## Redaction Relationship

In [ ]:
# Redaction rate by category (fraction of docs with any redaction), top 25
redaction_rate = df.groupby("granular_category")["has_any_redaction"].mean().sort_values(ascending=False)
top25_rate = redaction_rate.head(25)

fig, ax = plt.subplots(figsize=(12, 8))
top25_rate.iloc[::-1].plot.barh(ax=ax, color=sns.color_palette(PALETTE)[2], edgecolor="black")
ax.set_title("Redaction Rate by Granular Category (Top 25)")
ax.set_xlabel("Fraction of Documents with Any Redaction")
ax.set_ylabel("Granular Category")
plt.tight_layout()
plt.show()

In [ ]:
# Mean coverage percent by category (docs with redactions only), top 25
redacted = df[df["has_any_redaction"] == True]
mean_cov = redacted.groupby("granular_category")["mean_coverage_percent"].mean().sort_values(ascending=False)
top25_cov = mean_cov.head(25)

fig, ax = plt.subplots(figsize=(12, 8))
top25_cov.iloc[::-1].plot.barh(ax=ax, color=sns.color_palette(PALETTE)[3], edgecolor="black")
ax.set_title("Mean Coverage Percent by Granular Category (Top 25, Redacted Docs Only)")
ax.set_xlabel("Mean Coverage (%)")
ax.set_ylabel("Granular Category")
plt.tight_layout()
plt.show()

In [ ]:
# Mean total detections by category, top 25
mean_det = redacted.groupby("granular_category")["total_detections"].mean().sort_values(ascending=False)
top25_det = mean_det.head(25)

fig, ax = plt.subplots(figsize=(12, 8))
top25_det.iloc[::-1].plot.barh(ax=ax, color=sns.color_palette(PALETTE)[4], edgecolor="black")
ax.set_title("Mean Total Detections by Granular Category (Top 25, Redacted Docs Only)")
ax.set_xlabel("Mean Total Detections")
ax.set_ylabel("Granular Category")
plt.tight_layout()
plt.show()

## Length Relationship

In [ ]:
# Median page count by category, top 25
median_pages = df.groupby("granular_category")["total_pages"].median().sort_values(ascending=False)
top25_pages = median_pages.head(25)

fig, ax = plt.subplots(figsize=(12, 8))
top25_pages.iloc[::-1].plot.barh(ax=ax, color=sns.color_palette(PALETTE)[5], edgecolor="black")
ax.set_title("Median Page Count by Granular Category (Top 25)")
ax.set_xlabel("Median Total Pages")
ax.set_ylabel("Granular Category")
plt.tight_layout()
plt.show()

In [ ]:
# Box plot of total_pages for top 15 categories (by doc count)
top15_cats = cat_counts.head(15).index.tolist()
plot_df = df[df["granular_category"].isin(top15_cats)].copy()
plot_df["granular_category"] = pd.Categorical(
    plot_df["granular_category"], categories=top15_cats, ordered=True
)

fig, ax = plt.subplots(figsize=(14, 8))
sns.boxplot(
    data=plot_df, y="granular_category", x="total_pages",
    palette=PALETTE, ax=ax,
)
ax.set_title("Distribution of Total Pages by Granular Category (Top 15 by Count)")
ax.set_xlabel("Total Pages")
ax.set_ylabel("Granular Category")
plt.tight_layout()
plt.show()